In [9]:
import pandas as pd
import glob
import json
import ast
import os

In [ ]:
# Configurações
tamanho_lote = 100  # 100 variáveis por ficheiro é um número seguro para o Gemini
diretorio_saida = 'lotes_v2'
os.makedirs(diretorio_saida, exist_ok=True)

# Prompt que será escrito no início de cada ficheiro
prompt_classificacao = """
You are a strict data engineering system. Classify the exact database variables provided into ONE of the permitted categories. You MUST strictly follow these exact definitions:

PERMITTED CATEGORIES AND STRICT RULES:
- SOCIOECONOMIC: Home ownership, number of children/dependents, household size, length of employment, employment status, occupation, family income, income, wealth, parents’ income, social class, education.
- DEMOGRAPHIC: Age, gender, ethnicity, marital status, family life cycle. (CRITICAL: DO NOT put geographical locations here).
- VALUES, ATTITUDES and BEHAVIORAL: Socio-motivation, attitudes toward debt, perceived financial wellbeing, religious practices, consumer behavior, spending pattern, attitudes toward money, risk-taking, compulsive buying, delay of gratification, financial knowledge.
- INSTITUTIONAL and FINANCIAL: Number of debts, length of relationship with the bank, number of bank accounts, debt to income ratio, total financial assets, payment pattern, credit limit, existing credit commitments, credit score, number of credit cards, past credit history, loan amount, taking debt advice, loan duration, account balance, purpose of loan.
- PERSONALITY: Self-control, emotional stability, intelligence, optimism, extraversion, impulsiveness.
- SITUATIONAL: STRICTLY Adverse life events or life-altering events. (CRITICAL: DO NOT put standard dates, terms, or housing here).
- EDUCATIONAL: Field of study, GPA (Grade Point Average), year at school. (General education level goes to SOCIOECONOMIC).
- MACROECONOMIC: General economic indicators (inflation, GDP).
- HEALTH-RELATED: Physical and mental health indicators.
- ALTERNATIVE: Social network patterns, posting time, friends, daily calls, SMS patterns, disclosure of social media profile, and geographical locations (city, zipcode, region, country, IP).
- UNCLASSIFIED: System IDs, gibberish, transaction IDs, passwords, hashes, pure system metadata.

OUTPUT RULES:
1. Return ONLY a JSON object.
2. The keys MUST be the exact variable names provided.
3. The values MUST be the exact permitted category strings.
4. One key-value pair per variable.

RAW VARIABLES TO CLASSIFY:
"""

df = pd.read_excel(os.path.join('../Data/PreProcessed', '06_Datasets.xlsx'))
df_clear = df[df['is_encrypted'] == 0].copy()
df_clear['Columns'] = df_clear['Columns'].apply(ast.literal_eval)

df_vars = df_clear[['id', 'Columns']].explode('Columns').dropna()

df_vars = df_vars.drop_duplicates(subset=['id', 'Columns'])

lista_vars_com_id = df_vars.values.tolist()

for i in range(0, len(lista_vars_com_id), tamanho_lote):
    lote_atual = lista_vars_com_id[i:i + tamanho_lote]
    num_lote = (i // tamanho_lote) + 1
    nome_ficheiro = f"{diretorio_saida}/lote_{num_lote}.md"
    
    with open(nome_ficheiro, 'w', encoding='utf-8') as f:
        f.write(prompt_classificacao)
        for id_dataset, var in lote_atual:
            f.write(f"- ID: {id_dataset} | Variable: {var}\n")

print(f"Sucesso! {num_lote} ficheiros criados na pasta '{diretorio_saida}'.")

Sucesso! 9 ficheiros criados na pasta 'lotes_v2'.


In [ ]:
diretorio_jsons = 'respostas'
dicionario_consolidado = {}

caminhos_jsons = glob.glob(os.path.join(diretorio_jsons, 'r*.json'))

for caminho in caminhos_jsons:
    with open(caminho, 'r', encoding='utf-8') as f:
        try:
            dados_lote = json.load(f)
            dicionario_consolidado.update(dados_lote)
        except json.JSONDecodeError:
            print(f"Erro de sintaxe no ficheiro: {caminho}. Verifique se copiou as chavetas corretamente.")

ficheiro_final = 'classificacao_final_consolidada.json'
with open(ficheiro_final, 'w', encoding='utf-8') as f:
    json.dump(dicionario_consolidado, f, indent=4, ensure_ascii=False)

print(f"Fusão concluída! {len(dicionario_consolidado)} variáveis agrupadas no ficheiro '{ficheiro_final}'.")

Fusão concluída! 786 variáveis agrupadas no ficheiro 'classificacao_final_consolidada.json'.
